In [1]:
# Cell 1: Imports & Configuration
import os
import re
import io
import requests
import zipfile
import pandas as pd
import numpy as np
import logging
from datetime import datetime, timedelta
from pandas_datareader import data as pdr

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- CONFIGURATION (The "Truth" Filters) ---

# 1. STRICT THEMES: Only trust rows that GDELT explicitly flagged as financial
FINANCIAL_THEMES = [
    'ECON_STOCKMARKET', 'EPU_POLICY', 'EPU_POLICY_BUDGET', 
    'EPU_POLICY_CENTRAL_BANK', 'FINANCE', 'TAX_ECON_PRICE'
]

# 2. FLEXIBLE ALIASES: Capture all variations of the entity
ENTITY_CONFIG = {
    'AAPL': {
        'aliases': ['apple inc', 'apple computer', 'apple store', 'apple tv', 'aapl'],
        'blacklist': ['apple valley', 'little apple', 'recipe', 'bake', 'pie', 'cider', 'yeast']
    },
    'AMZN': {
        'aliases': ['amazon.com', 'amazon inc', 'aws', 'amazon web services'],
        'blacklist': ['amazon river', 'forest']
    }
}

# 3. STORAGE PATHS
DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
# Cell 2: Ingestion Logic (Updated for Stability & Column Selection)

def download_daily_gkg(date_str):
    """
    Downloads and parses a single day of GKG data.
    Uses the user's specific 11-column layout for stability.
    """
    url = f"http://data.gdeltproject.org/gkg/{date_str}.gkg.csv.zip"
    logger.info(f"Fetching GKG: {url}")
    
    try:
        r = requests.get(url)
        if r.status_code != 200:
            logger.warning(f"URL not found: {url}")
            return pd.DataFrame()
            
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                # 1. Define the 11 columns found in the file (Bronze Layer Schema)
                bronze_cols = [
                    'DATE', 'NUMARTS', 'COUNTS', 'THEMES', 'LOCATIONS', 
                    'PERSONS', 'ORGANIZATIONS', 'TONE', 'CAMEOEVENTIDS', 
                    'SOURCES', 'SOURCEURLS'
                ]
                
                # 2. Load the WHOLE file first for stability (avoid usecols errors)
                # use sep='\t' for GDELT, on_bad_lines='skip' to handle malformed rows
                df = pd.read_csv(f, sep='\t', header=None, names=bronze_cols, 
                                 encoding='utf-8', on_bad_lines='skip', dtype=str)
                
                # 3. Pick only the needed columns (as requested)
                # SOURCEURLS is the link, ORGANIZATIONS is the entity, THEMES is the context
                needed_cols = ['DATE', 'THEMES', 'ORGANIZATIONS', 'TONE', 'SOURCEURLS']
                
                # Drop rows where critical info is missing
                df = df.dropna(subset=['SOURCEURLS', 'ORGANIZATIONS'])
                
                return df[needed_cols]
                
    except Exception as e:
        logger.error(f"Download failed for {date_str}: {e}")
        return pd.DataFrame()

def ingest_stooq_data(ticker, start_date):
    """Fetch stock data from Stooq"""
    try:
        df = pdr.DataReader(ticker, 'stooq', start_date, datetime.now())
        df = df.reset_index().rename(columns={'Date': 'date', 'Close': 'close', 'Volume': 'volume', 'Open': 'open', 'High': 'high', 'Low': 'low'})
        df['date'] = pd.to_datetime(df['date']).dt.date
        df['ticker'] = ticker
        return df
    except Exception as e:
        logger.error(f"Stooq failed for {ticker}: {e}")
        return pd.DataFrame()

In [3]:
# Cell 3: Silver Processing (Keyword Themes + Slug Extraction)

def extract_slug(url):
    """
    Extracts a clean title from the URL slug.
    """
    if not isinstance(url, str): return ""
    
    # Regex to find the last segment before .html or query params
    match = re.search(r'([^/]+)(?:\.html|\?)?$', url.strip('/'))
    if match:
        slug = match.group(1)
        slug = re.sub(r'\d+', '', slug) # Remove ID numbers
        slug = slug.replace('-', ' ').replace('_', ' ').replace('+', ' ')
        return slug.strip().title()
    return ""

def filter_and_clean(df, ticker):
    """
    Applies Keyword Theme Filter + Flexible Entity Filter.
    """
    config = ENTITY_CONFIG[ticker]
    
    # 1. THEME KEYWORDS (The "Broad Truth")
    # Instead of 'ECON_STOCKMARKET', we just look for 'ECON' or 'FIN'
    theme_keywords = ['ECON', 'FIN', 'TAX', 'BANK', 'BUDGET', 'TRADE', 'MKT']
    theme_pattern = '|'.join(theme_keywords)
    
    # Filter: Keep rows where THEMES column contains any of the keywords
    df_filtered = df[df['THEMES'].astype(str).str.contains(theme_pattern, na=False)].copy()
    
    # 2. Flexible Entity Filter (Search in ORGANIZATIONS or SOURCEURLS)
    alias_pattern = '|'.join(config['aliases'])
    mask_org = df_filtered['ORGANIZATIONS'].astype(str).str.contains(alias_pattern, case=False, na=False)
    mask_url = df_filtered['SOURCEURLS'].astype(str).str.contains(alias_pattern, case=False, na=False)
    df_filtered = df_filtered[mask_org | mask_url]
    
    # 3. Blacklist Filter
    if config['blacklist']:
        blacklist_pattern = '|'.join(config['blacklist'])
        # Note: Using SOURCEURLS now
        mask_noise = df_filtered['SOURCEURLS'].astype(str).str.contains(blacklist_pattern, case=False, na=False)
        df_filtered = df_filtered[~mask_noise]
        
    # 4. Extract Slug using SOURCEURLS
    df_filtered['extracted_title'] = df_filtered['SOURCEURLS'].apply(extract_slug)
    
    # Remove empty/short titles
    df_filtered = df_filtered[df_filtered['extracted_title'].str.len() > 10]
    
    # Rename for consistency with downstream tasks
    df_filtered['ticker'] = ticker
    df_filtered = df_filtered.rename(columns={'SOURCEURLS': 'DocumentIdentifier', 'TONE': 'V2Tone'})
    
    return df_filtered[['DATE', 'ticker', 'extracted_title', 'V2Tone', 'DocumentIdentifier']]

# --- EXECUTION BLOCK ---
# 1. Select your Date Range (Format: YYYY-MM-DD)
START_DATE = '2025-12-18'
END_DATE   = '2025-12-20'

# 2. Generate list of dates
start_d = datetime.strptime(START_DATE, '%Y-%m-%d')
end_d = datetime.strptime(END_DATE, '%Y-%m-%d')
delta = end_d - start_d
dates = [(start_d + timedelta(days=i)).strftime('%Y%m%d') for i in range(delta.days + 1)]

print(f"Target Dates: {dates}")

all_news = []
for d in dates:
    raw_df = download_daily_gkg(d)
    if not raw_df.empty:
        processed = filter_and_clean(raw_df, 'AAPL')
        all_news.append(processed)

if all_news:
    df_silver_news = pd.concat(all_news, ignore_index=True)
    df_silver_news['date_obj'] = pd.to_datetime(df_silver_news['DATE'], format='%Y%m%d', errors='coerce').dt.date
    if df_silver_news['date_obj'].isnull().any():
         df_silver_news['date_obj'] = pd.to_datetime(df_silver_news['DATE'], format='%Y%m%d%H%M%S', errors='coerce').dt.date

    print(f"Silver Layer: Cleaned {len(df_silver_news)} news items for the selected range.")
    print(df_silver_news.head())
else:
    print(f"No relevant news found between {START_DATE} and {END_DATE}.")

2025-12-22 09:03:06,235 - INFO - Fetching GKG: http://data.gdeltproject.org/gkg/20251218.gkg.csv.zip


Target Dates: ['20251218', '20251219', '20251220']


2025-12-22 09:03:15,409 - INFO - Fetching GKG: http://data.gdeltproject.org/gkg/20251219.gkg.csv.zip
2025-12-22 09:03:24,051 - INFO - Fetching GKG: http://data.gdeltproject.org/gkg/20251220.gkg.csv.zip


Silver Layer: Cleaned 43 news items for the selected range.
       DATE ticker                                    extracted_title  \
0  20251218   AAPL  How Openai Used Aggressive Discounts To Domina...   
1  20251218   AAPL  Marketminute    The Great Reversal Why Citigro...   
2  20251218   AAPL  Apple Makeschanges To Ios Software In Face Of ...   
3  20251218   AAPL  Marketminute    Amazons Ai Fueled Ascent Navig...   
4  20251218   AAPL  Teslas Musk Premium In Focus With Spacex Ipo O...   

                                              V2Tone  \
0  0.477164280845262,2.86298568507157,2.385821404...   
1  0.435729847494554,3.7763253449528,3.3405954974...   
2  -1.1965811965812,2.22222222222222,3.4188034188...   
3  0.735895339329518,3.67947669664759,2.943581357...   
4  0.333055786844296,2.99750208159867,2.664446294...   

                                  DocumentIdentifier    date_obj  
0  https://www.latimes.com/business/story/2025-12...  2025-12-18  
1  https://markets.financialco

In [4]:
# Cell 4: NLP Preparation (Output 1)

def prepare_nlp_input(df):
    """
    Creates the exact input structure for your T5/ABSA model.
    Adds a placeholder for the result.
    """
    nlp_df = df.copy()
    
    # Instruction for T5 (if using instruct model)
    # Format: "Aspect: [Entity] Context: [Title]"
    nlp_df['nlp_instruction'] = (
        "Analyze sentiment for " + nlp_df['ticker'] + 
        " in text: " + nlp_df['extracted_title']
    )
    
    # Placeholder for your "Own Sentiment Score"
    nlp_df['predicted_sentiment_score'] = np.nan # To be filled by model
    
    return nlp_df[['date_obj', 'ticker', 'extracted_title', 'nlp_instruction', 'predicted_sentiment_score']]

df_nlp_input = prepare_nlp_input(df_silver_news)

# Save for your NLP model to pick up
df_nlp_input.to_csv(f'{DATA_DIR}/nlp_input_ready.csv', index=False)
print("Saved nlp_input_ready.csv for Model Inference.")

Saved nlp_input_ready.csv for Model Inference.


In [ ]:
# Cell 5: SIMULATION of NLP Inference (To allow building the next steps)
# In reality, you would load 'nlp_input_ready.csv', run your model, and fill the NaN column.
# Here we simulate it so we can write the ML training logic.

logger.info("Simulating NLP inference...")
# Simulate scores between -1 (Negative) and 1 (Positive)
df_nlp_input['predicted_sentiment_score'] = np.random.uniform(-1, 1, size=len(df_nlp_input))

# Also normalize GDELT's native tone to comparable scale (-10 to 10 -> -1 to 1)
# We handle GDELT tone parsing here strictly for comparison/fallback
def parse_gdelt_tone(tone_str):
    try:
        return float(str(tone_str).split(',')[0]) / 10.0
    except:
        return 0.0

df_silver_news['gdelt_score'] = df_silver_news['V2Tone'].apply(parse_gdelt_tone)

In [ ]:
# Cell 6: Gold Layer - Aggregation & ML Feature Engineering (Output 2)

def create_ml_training_data(news_df, nlp_df, ticker):
    # 1. Fetch Stock Data
    start_date = news_df['date_obj'].min() - timedelta(days=30) # Buffer for moving averages
    stock_df = ingest_stooq_data(ticker, start_date)
    
    if stock_df.empty:
        logger.error("No stock data found.")
        return pd.DataFrame()

    # 2. Aggregate Sentiment by Day
    # We aggregate both our NEW NLP score and the old GDELT score for feature richness
    daily_sentiment = nlp_df.groupby('date_obj')['predicted_sentiment_score'].agg(['mean', 'count', 'std']).reset_index()
    daily_sentiment.columns = ['date', 'my_sentiment_avg', 'news_volume', 'sentiment_volatility']
    
    # 3. Merge Stock & Sentiment
    # Left join on Stock dates (Market is closed weekends, news isn't. We adhere to market days)
    gold_df = pd.merge(stock_df, daily_sentiment, on='date', how='left')
    
    # Fill missing sentiment (no news days) with 0 or forward fill
    gold_df[['my_sentiment_avg', 'news_volume', 'sentiment_volatility']] = gold_df[['my_sentiment_avg', 'news_volume', 'sentiment_volatility']].fillna(0)
    
    # 4. Feature Engineering for ML
    gold_df = gold_df.sort_values('date')
    
    # Target: Next Day Return (Shifted back by 1)
    gold_df['daily_return'] = gold_df['close'].pct_change()
    gold_df['target_next_day_return'] = gold_df['daily_return'].shift(-1)
    
    # Features: Moving Averages
    gold_df['ma_7'] = gold_df['close'].rolling(window=7).mean()
    gold_df['ma_30'] = gold_df['close'].rolling(window=30).mean()
    
    # Features: Lagged Sentiment (Yesterday's news predicts today's open)
    gold_df['lag_1_sentiment'] = gold_df['my_sentiment_avg'].shift(1)
    
    # Drop NaN created by lags/rolling
    gold_df = gold_df.dropna()
    
    return gold_df

# Execute for AAPL
df_ml_ready = create_ml_training_data(df_silver_news, df_nlp_input, 'AAPL')

if not df_ml_ready.empty:
    df_ml_ready.to_csv(f'{DATA_DIR}/ml_training_data_aapl.csv', index=False)
    print("✓ Saved ml_training_data_aapl.csv")
    print("\nSample Training Data:")
    print(df_ml_ready[['date', 'close', 'my_sentiment_avg', 'lag_1_sentiment', 'target_next_day_return']].tail())
else:
    print("Failed to generate ML data.")